# RepDistiller (CRD): Hyperparameter Sweeps & Analysis on Google Colab

This notebook runs and analyzes **Contrastive Representation Distillation (CRD)** experiments across:
1. **Vision (CIFAR-100):** `resnet32x4` (teacher) $\rightarrow$ `resnet8x4` (student) — sweeping temperature $\tau$ (`--nce_t`) and contrastive loss weight $\beta$ (`-b`).
2. **NLP (SST-2, optional):** `bert-base-uncased` (teacher) $\rightarrow$ 2-layer Transformer or BiLSTM (student) — testing whether contrastive `[CLS]` representation distillation transfers to text classification.

### Before you start
1. Enable GPU in Colab: **Runtime → Change runtime type → T4 GPU** (or L4/A100 on Colab Pro).
2. Upload `RepDistiller.zip` to Google Drive (`MyDrive/RepDistiller.zip`, excluding `.venv/`), **or** set `GIT_URL` in Step 2.

## Step 1 & 2: Environment Verification, Google Drive Persistence & Dependencies
**Why:** Colab runtimes are ephemeral and disconnect after inactivity. Mounting Google Drive lets us back up every trial immediately after completion and resume seamlessly if a session drops.

In [ ]:
# Step 1: Verify GPU & mount Google Drive
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Unpack/clone RepDistiller into /content/RepDistiller and install packages
import os

ZIP_PATH = "/content/drive/MyDrive/RepDistiller.zip"  # Option A: zip in Google Drive
GIT_URL = "https://github.com/gmbcode/RepDistiller_Alpha.git"  # Option B: GitHub repo (if private, use PAT or ZIP_PATH)

if not os.path.exists("/content/RepDistiller"):
    if GIT_URL:
        !git clone {GIT_URL} /content/RepDistiller
    elif os.path.exists(ZIP_PATH):
        !unzip -q "{ZIP_PATH}" -d /content/
    else:
        raise FileNotFoundError(
            f"Neither {ZIP_PATH} nor GIT_URL was provided. Upload RepDistiller.zip to MyDrive or set GIT_URL."
        )

%cd /content/RepDistiller
!pip install -q transformers datasets tqdm pandas matplotlib

## Step 3 & 4: Pretrained Teacher Setup & Dry-Run Sanity Check
**Why we run these checks first:**
- **Pretrained Teacher (`resnet32x4`):** Distillation needs a converged fixed teacher (`79.42%` CIFAR-100 accuracy in the CRD paper). Skipping teacher retraining isolates student distillation performance.
- **Dry-Run (`--dry_run`):** Validates all 25 CLI commands, `dataset_patch` (`--subset_frac`), and checkpoint paths in seconds before committing GPU hours.

In [ ]:
# Step 3: Fast setup for Teacher Checkpoint & CIFAR-100 (Drive cache + HuggingFace mirrors)
import hashlib
import os
import shutil
import subprocess
import tarfile
import zipfile
import torch
import torchvision.datasets as tv_datasets

DRIVE_CACHE_DIR = "/content/drive/MyDrive/crd_sweep_results"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

# ── 3A. Teacher checkpoint (resnet32x4_vanilla) ─────────────────────────────
teacher_dir = "./save/models/resnet32x4_vanilla"
teacher_ckpt = os.path.join(teacher_dir, "ckpt_epoch_240.pth")
drive_teacher = os.path.join(DRIVE_CACHE_DIR, "resnet32x4_ckpt_epoch_240.pth")
os.makedirs(teacher_dir, exist_ok=True)

if os.path.exists(teacher_ckpt):
    print("Teacher checkpoint already present:", teacher_ckpt)
elif os.path.exists(drive_teacher):
    shutil.copy2(drive_teacher, teacher_ckpt)
    print("Restored teacher checkpoint from Drive cache:", teacher_ckpt)
else:
    print("Downloading resnet32x4_vanilla teacher with resume/retry...")
    subprocess.run([
        "curl", "-fSL", "--retry", "8", "--retry-all-errors", "--retry-delay", "3", "-C", "-",
        "http://shape2prog.csail.mit.edu/repo/resnet32x4_vanilla/ckpt_epoch_240.pth",
        "-o", teacher_ckpt,
    ], check=True)
    shutil.copy2(teacher_ckpt, drive_teacher)
    print("Cached teacher checkpoint to Drive:", drive_teacher)

_ckpt = torch.load(teacher_ckpt, map_location="cpu", weights_only=False)
assert isinstance(_ckpt, dict) and "model" in _ckpt, "Unexpected teacher checkpoint format"
print(f"Verified teacher checkpoint OK ({os.path.getsize(teacher_ckpt):,} bytes)")

# ── 3B. CIFAR-100 dataset (Local -> Drive/Kaggle Zip -> HuggingFace CDN -> Toronto fallback)
CIFAR_MD5 = "eb9058c3a382ffc7106e4002c42a8d85"
os.makedirs("./data", exist_ok=True)
local_tgz = "./data/cifar-100-python.tar.gz"
drive_tgz = os.path.join(DRIVE_CACHE_DIR, "cifar-100-python.tar.gz")
KAGGLE_ZIPS = [
    "/content/drive/MyDrive/cifar100.zip",
    "/content/drive/MyDrive/cifar-100-python.zip",
    "/content/cifar100.zip",
]

def md5_ok(path, expected):
    if not os.path.exists(path):
        return False
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest() == expected

def cifar_ready():
    try:
        tv_datasets.CIFAR100(root="./data", train=True, download=False)
        tv_datasets.CIFAR100(root="./data", train=False, download=False)
        return True
    except Exception:
        return False

if not cifar_ready():
    # 1) Try Kaggle zip on Drive/Colab if present
    for kz in KAGGLE_ZIPS:
        if os.path.exists(kz):
            print("Found Kaggle zip, extracting:", kz)
            with zipfile.ZipFile(kz, "r") as zf:
                names = zf.namelist()
                if any(n.startswith("cifar-100-python/") for n in names) or "cifar-100-python.tar.gz" in names:
                    zf.extractall("./data")
                elif all(x in names for x in ["train", "test", "meta"]):
                    os.makedirs("./data/cifar-100-python", exist_ok=True)
                    zf.extractall("./data/cifar-100-python")
            if os.path.exists(local_tgz) and md5_ok(local_tgz, CIFAR_MD5):
                with tarfile.open(local_tgz, "r:gz") as tf:
                    tf.extractall("./data")
            if cifar_ready():
                break

    # 2) Try Drive-cached tar.gz
    if not cifar_ready() and md5_ok(drive_tgz, CIFAR_MD5):
        print("Restoring CIFAR-100 from Drive cache...")
        shutil.copy2(drive_tgz, local_tgz)
        with tarfile.open(local_tgz, "r:gz") as tf:
            tf.extractall("./data")

    # 3) Try fast HuggingFace CDN mirrors (same exact archive MD5)
    if not cifar_ready():
        hf_urls = [
            "https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz",
            "https://huggingface.co/datasets/xingslong/cifar-100-python/resolve/main/cifar-100-python.tar.gz",
        ]
        for url in hf_urls:
            print("Downloading CIFAR-100 from HuggingFace CDN:", url)
            res = subprocess.run(["curl", "-fSL", "--retry", "3", url, "-o", local_tgz])
            if res.returncode == 0 and md5_ok(local_tgz, CIFAR_MD5):
                with tarfile.open(local_tgz, "r:gz") as tf:
                    tf.extractall("./data")
                shutil.copy2(local_tgz, drive_tgz)
                print("Extracted & cached CIFAR-100 to Drive:", drive_tgz)
                break
            if os.path.exists(local_tgz):
                os.remove(local_tgz)

    # 4) Last resort: torchvision default (cs.toronto.edu)
    if not cifar_ready():
        print("Falling back to torchvision default CIFAR-100 download...")
        tv_datasets.CIFAR100(root="./data", train=True, download=True)
        tv_datasets.CIFAR100(root="./data", train=False, download=True)

assert cifar_ready(), "CIFAR-100 failed integrity check!"
print("CIFAR-100 verified and ready in ./data")

In [ ]:
# Step 4: Dry-run check of the 5x5 grid commands
!python sweep_tau_beta.py --dry_run --num_workers 2

## Step 5: Experiment 1 — Vision CRD $\tau \times \beta$ Grid Sweep (CIFAR-100)

### What we are testing & why
CRD optimizes $\mathcal{L} = r\,\mathcal{L}_{\text{CE}} + a\,\mathcal{L}_{\text{KD}} + \beta\,\mathcal{L}_{\text{CRD}}$. Here we fix $r = 1.0$ and $a = 0.0$ (**pure CRD**, no logit-KD mixing) and sweep the two core contrastive hyperparameters around the paper's defaults ($\tau = 0.07, \beta = 0.8$):
- **NCE Temperature $\tau$ (`--nce_t` $\in \{0.03, 0.07, 0.14, 0.30, 0.60\}$):** Controls how sharply the contrastive objective separates positive teacher–student pairs from memory-bank negatives.
  - *Hypothesis:* Very small $\tau$ (`0.03`) puts extreme weight on hard negatives and can destabilize early training; very large $\tau$ (`0.30`, `0.60`) flattens the similarity distribution and weakens representational transfer.
- **CRD Loss Weight $\beta$ (`-b` $\in \{0.2, 0.4, 0.8, 1.2, 1.6\}$):** Controls the trade-off between supervised label cross-entropy and contrastive teacher–student feature alignment.
  - *Hypothesis:* Too small $\beta$ (`0.2`) under-utilizes the teacher's representation geometry; too large $\beta$ (`1.6`) can compete with class-discriminative cross-entropy gradients.
- **Proxy horizon (`EPOCHS=30`, `SUBSET_FRAC=1.0`):** Full CRD training uses 240 epochs; 30 epochs provides a fast, reliable ranking signal across the 25 combinations (set `SUBSET_FRAC=0.3` for ~3x faster exploratory runs on a free T4).

In [ ]:
# Step 5: Run CIFAR-100 CRD tau x beta sweep (parallelized + tqdm + resume & Drive backup)
import csv
import itertools
import os
import shutil
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from types import SimpleNamespace
from tqdm.auto import tqdm

import sweep_tau_beta as stb

TAU_VALUES = [0.03, 0.07, 0.14, 0.30, 0.60]
BETA_VALUES = [0.2, 0.4, 0.8, 1.2, 1.6]
EPOCHS = 30          # 30 for proxy sweep; 240 for full paper schedule
SUBSET_FRAC = 1.0    # Set to 0.3 for ~3x faster proxy runs
MAX_PARALLEL = 2     # Concurrent runs on 1 GPU (2-3 optimal on L4 with 12 vCPUs / 22.5 GB VRAM)
NUM_WORKERS = 4      # DataLoader workers per run (keep MAX_PARALLEL * NUM_WORKERS <= 12)
TEACHER_PATH = "./save/models/resnet32x4_vanilla/ckpt_epoch_240.pth"
DRIVE_RESULTS = "/content/drive/MyDrive/crd_sweep_results"

os.makedirs(stb.OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs("./save/student_model", exist_ok=True)
os.makedirs("./save/student_tensorboards", exist_ok=True)

drive_csv = os.path.join(DRIVE_RESULTS, "results.csv")
completed = {}
if os.path.exists(drive_csv):
    shutil.copy2(drive_csv, stb.RESULTS_CSV)
    with open(stb.RESULTS_CSV, newline="") as f:
        for row in csv.DictReader(f):
            if row.get("status") == "ok":
                completed[(float(row["tau"]), float(row["beta"]))] = row
    print(f"Resuming: found {len(completed)} completed runs in {drive_csv}")
else:
    with open(stb.RESULTS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["trial_id", "tau", "beta", "best_acc", "status", "timestamp"])

sweep_args = SimpleNamespace(
    epochs=EPOCHS,
    subset_frac=SUBSET_FRAC,
    num_workers=NUM_WORKERS,
    teacher_path=TEACHER_PATH,
    gpu=0,
)

grid = list(itertools.product(TAU_VALUES, BETA_VALUES))
pending = [(tid, tau, beta) for tid, (tau, beta) in enumerate(grid, start=1) if (float(tau), float(beta)) not in completed]
print(f"Total: {len(grid)} | Skipping completed: {len(grid) - len(pending)} | Running: {len(pending)} (MAX_PARALLEL={MAX_PARALLEL})")

# Clean up any orphaned train_student.py processes from an earlier interrupted run
subprocess.run(["pkill", "-9", "-f", "train_student.py"], check=False)

csv_lock = threading.Lock()
stop_event = threading.Event()
active_procs = set()
pbar = tqdm(total=len(grid), initial=len(grid) - len(pending), desc="CRD tau-beta sweep")

def run_one(trial_id, tau, beta):
    if stop_event.is_set():
        return None
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cmd = stb.build_command(tau, beta, trial_id, sweep_args)
    tag = f"[Run {trial_id}/{len(grid)} | τ={tau}, β={beta}]"
    tqdm.write(f"{tag} started")

    log_lines = []
    acc_lines = 0
    val_epoch = 0
    seen_teacher = False
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with csv_lock:
        active_procs.add(proc)
    try:
        for line in proc.stdout:
            if stop_event.is_set():
                break
            log_lines.append(line)
            if "teacher accuracy:" in line:
                seen_teacher = True
                tqdm.write(f"{tag} {line.rstrip()}")
            elif "* Acc@1" in line:
                acc_lines += 1
                if seen_teacher and acc_lines % 2 == 1:
                    val_epoch += 1
                    if val_epoch % 5 == 0 or val_epoch == 1 or val_epoch == EPOCHS:
                        tqdm.write(f"{tag} Ep {val_epoch}/{EPOCHS}: {line.strip()}")
        ret = proc.wait()
    finally:
        with csv_lock:
            active_procs.discard(proc)

    if stop_event.is_set():
        return None

    log_text = "".join(log_lines)
    acc = stb.extract_accuracy(log_text)
    status = "ok" if ret == 0 else f"exit{ret}"

    log_name = f"log_tau{tau}_beta{beta}.txt"
    local_log = os.path.join(stb.OUTPUT_DIR, log_name)
    with open(local_log, "w", encoding="utf-8") as lf:
        lf.write(log_text)

    with csv_lock:
        with open(stb.RESULTS_CSV, "a", newline="") as f:
            csv.writer(f).writerow([trial_id, tau, beta, acc, status, timestamp])
        shutil.copy2(stb.RESULTS_CSV, drive_csv)
        shutil.copy2(local_log, os.path.join(DRIVE_RESULTS, log_name))
        pbar.set_postfix(last=f"τ={tau},β={beta}:{acc}")
        pbar.update(1)

    tqdm.write(f"{tag} -> best_acc={acc} status={status} (backed up to Drive)")
    if status != "ok":
        tail = "\n".join(log_lines[-15:])
        tqdm.write(f"{tag} [ERROR tail]:\n{tail}")
    return trial_id, tau, beta, acc, status

pool = ThreadPoolExecutor(max_workers=MAX_PARALLEL)
try:
    futures = [pool.submit(run_one, tid, tau, beta) for tid, tau, beta in pending]
    for fut in as_completed(futures):
        fut.result()
    pool.shutdown(wait=True)
except KeyboardInterrupt:
    stop_event.set()
    with csv_lock:
        for p in list(active_procs):
            p.kill()
    pool.shutdown(wait=False, cancel_futures=True)
    raise
finally:
    pbar.close()
stb.plot_heatmap()
heatmap_local = os.path.join(stb.OUTPUT_DIR, "heatmap_tau_beta.png")
if os.path.exists(heatmap_local):
    shutil.copy2(heatmap_local, os.path.join(DRIVE_RESULTS, "heatmap_tau_beta.png"))

print("Done! All sweep results saved in:", DRIVE_RESULTS)

## Step 6: Visualize & Interpret Vision Sweep Results

The cell below generates three complementary views from `sweep_results/` and prints an automated summary:
1. **2D Accuracy Heatmap (`τ` vs `β`):** Identifies the optimal operating region and whether performance forms a smooth ridge or a sharp peak.
2. **1D Marginal Sensitivity Plots (`Acc@1 vs τ` and `Acc@1 vs β`):** Separates which hyperparameter drives larger variance and whether trends are monotonic or U-shaped.
3. **Epoch-by-Epoch Learning Curves (Best vs Paper Default `τ=0.07, β=0.8` vs Worst):** Distinguishes *optimization instability* (noisy/diverging curves) from *under-distillation* (smooth but lower ceiling).

In [ ]:
# Step 6: Visualize Vision sweep results + automated quantitative interpretation
import os
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DRIVE_RESULTS = "/content/drive/MyDrive/crd_sweep_results"
os.makedirs("sweep_results", exist_ok=True)
csv_path = "sweep_results/results.csv" if os.path.exists("sweep_results/results.csv") else os.path.join(DRIVE_RESULTS, "results.csv")
df = pd.read_csv(csv_path)
ok_df = df[df["status"] == "ok"].copy()
ok_df["best_acc"] = pd.to_numeric(ok_df["best_acc"], errors="coerce")
ok_df = ok_df.dropna(subset=["best_acc"])

if ok_df.empty:
    raise RuntimeError("No successful runs found in sweep_results/results.csv yet.")

# Deduplicate keeping latest run per (tau, beta)
ok_df = ok_df.drop_duplicates(subset=["tau", "beta"], keep="last")
pivot = ok_df.pivot_table(index="tau", columns="beta", values="best_acc", aggfunc="mean")

# ── Figure 1: Heatmap + Marginal Sensitivity Curves ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: 2D Heatmap
im = axes[0].imshow(pivot.values, aspect="auto", cmap="viridis")
fig.colorbar(im, ax=axes[0], label="Test Acc@1 (%)")
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels([f"{b:g}" for b in pivot.columns])
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels([f"{t:g}" for t in pivot.index])
axes[0].set_xlabel("Beta (β, CRD weight)")
axes[0].set_ylabel("Tau (τ, NCE temperature)")
axes[0].set_title("Accuracy Heatmap (τ vs β)")
mean_v = np.nanmean(pivot.values)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            axes[0].text(j, i, f"{val:.2f}", ha="center", va="center",
                         color="white" if val < mean_v else "black", fontsize=9)
bi, bj = np.unravel_index(np.nanargmax(pivot.values), pivot.values.shape)
axes[0].add_patch(plt.Rectangle((bj - 0.5, bi - 0.5), 1, 1, fill=False, edgecolor="red", lw=2.5))

# Panel B: Sensitivity to Tau
for beta_val, grp in ok_df.sort_values("tau").groupby("beta"):
    axes[1].plot(grp["tau"], grp["best_acc"], marker="o", alpha=0.55, label=f"β={beta_val:g}")
tau_mean = ok_df.groupby("tau")["best_acc"].mean()
axes[1].plot(tau_mean.index, tau_mean.values, color="black", lw=2.5, marker="s", label="Mean across β")
axes[1].set_xscale("log")
axes[1].set_xlabel("Tau (τ, log scale)")
axes[1].set_ylabel("Test Acc@1 (%)")
axes[1].set_title("Sensitivity to Temperature τ")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

# Panel C: Sensitivity to Beta
for tau_val, grp in ok_df.sort_values("beta").groupby("tau"):
    axes[2].plot(grp["beta"], grp["best_acc"], marker="o", alpha=0.55, label=f"τ={tau_val:g}")
beta_mean = ok_df.groupby("beta")["best_acc"].mean()
axes[2].plot(beta_mean.index, beta_mean.values, color="black", lw=2.5, marker="s", label="Mean across τ")
axes[2].set_xlabel("Beta (β)")
axes[2].set_ylabel("Test Acc@1 (%)")
axes[2].set_title("Sensitivity to CRD Weight β")
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=8)

plt.tight_layout()
sens_path = "sweep_results/sensitivity_analysis.png"
plt.savefig(sens_path, dpi=150)
plt.show()

# ── Figure 2: Epoch Learning Curves (Best vs Default vs Worst) ──────────────
def parse_epoch_val_acc(tau, beta):
    fname = f"log_tau{tau}_beta{beta}.txt"
    log_path = os.path.join("sweep_results", fname)
    if not os.path.exists(log_path):
        log_path = os.path.join(DRIVE_RESULTS, fname)
    if not os.path.exists(log_path):
        return None
    txt = open(log_path, encoding="utf-8").read()
    if "teacher accuracy:" in txt:
        txt = txt.split("teacher accuracy:", 1)[1]
    accs = [float(x) for x in re.findall(r"\*\s+Acc@1\s+([\d.]+)", txt)]
    # train_distill prints * Acc@1 (even indices), validate prints * Acc@1 (odd indices)
    return accs[1::2] if len(accs) >= 2 else accs

best_row = ok_df.loc[ok_df["best_acc"].idxmax()]
worst_row = ok_df.loc[ok_df["best_acc"].idxmin()]
default_df = ok_df[(np.isclose(ok_df["tau"], 0.07)) & (np.isclose(ok_df["beta"], 0.8))]

candidates = [("Best", best_row), ("Worst", worst_row)]
if not default_df.empty:
    candidates.insert(1, ("Paper Default", default_df.iloc[0]))

fig2, ax2 = plt.subplots(figsize=(8, 4.5))
for tag, row in candidates:
    curve = parse_epoch_val_acc(row["tau"], row["beta"])
    if curve:
        ax2.plot(range(1, len(curve) + 1), curve, marker="o", markersize=3,
                 label=f"{tag}: τ={row['tau']:g}, β={row['beta']:g} (final={row['best_acc']:.2f}%)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Acc@1 (%)")
ax2.set_title("Convergence Comparison Across Epochs")
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
lc_path = "sweep_results/learning_curves_comparison.png"
plt.savefig(lc_path, dpi=150)
plt.show()

# Copy new figures to Drive
for p in [sens_path, lc_path]:
    if os.path.exists(p) and os.path.exists(DRIVE_RESULTS):
        shutil.copy2(p, os.path.join(DRIVE_RESULTS, os.path.basename(p)))

# ── Automated Quantitative Interpretation ───────────────────────────────────
print("=" * 68)
print("QUANTITATIVE INTERPRETATION SUMMARY")
print("=" * 68)
print(f"Completed runs     : {len(ok_df)} / {len(df)}")
print(f"Best config        : tau={best_row['tau']:g}, beta={best_row['beta']:g} -> {best_row['best_acc']:.2f}%")
print(f"Worst config       : tau={worst_row['tau']:g}, beta={worst_row['beta']:g} -> {worst_row['best_acc']:.2f}%")
if not default_df.empty:
    def_acc = float(default_df.iloc[0]["best_acc"])
    delta = float(best_row["best_acc"]) - def_acc
    print(f"Paper default      : tau=0.07, beta=0.8 -> {def_acc:.2f}% (best delta: {delta:+.2f}%)")

tau_range = float(tau_mean.max() - tau_mean.min())
beta_range = float(beta_mean.max() - beta_mean.min())
print(f"Mean Acc swing     : across tau = {tau_range:.2f}% | across beta = {beta_range:.2f}%")
if tau_range > beta_range * 1.25:
    print("Dominant factor    : Temperature (tau) drives larger sensitivity than beta.")
elif beta_range > tau_range * 1.25:
    print("Dominant factor    : Loss weight (beta) drives larger sensitivity than tau.")
else:
    print("Dominant factor    : Both tau and beta have comparable impact.")

print("\nTop 5 configurations:")
display(ok_df.sort_values("best_acc", ascending=False)[["trial_id", "tau", "beta", "best_acc"]].head(5))

### How to Interpret the Vision Sweep Patterns
- **Sharp drop at $\tau = 0.03$:** Temperature is too low — `exp(sim / τ)` amplifies small embedding noise into huge gradients on hard negatives, causing representation instability or slower early convergence (check the `Worst` learning curve for oscillation).
- **Gradual drop at $\tau \ge 0.30$:** Temperature is too high — cosine similarities look nearly uniform across positives and negatives, weakening contrastive mutual-information lower-bound pressure so the student behaves closer to vanilla CE.
- **Peak around $\tau \in [0.07, 0.14]$ & $\beta \in [0.4, 1.2]$:** Confirms the paper's sweet spot where positive teacher–student alignment is sharp without destabilizing SGD.
- **Drop at $\beta = 1.6$:** Contrastive loss overwhelms classification CE gradients, hurting final linear separability on CIFAR-100 labels.
- **Flat plateau across neighboring cells (within ~0.3–0.5%):** CRD is robust in that region; you can safely run a full **240-epoch** confirmation run (`!python train_student.py --path_t ... --distill crd --model_s resnet8x4 -a 0 -b <best_beta> --nce_t <best_tau> --epochs 240`) to compare against the paper's `75.51%` benchmark.

## Step 7 (Optional): Experiment 2 — NLP CRD Sweep (`train_nlp_crd.py` on SST-2)

### What we are testing & why
Unlike vision CRD (convolutional ResNets on CIFAR-100), [train_nlp_crd.py](train_nlp_crd.py) tests **cross-architecture NLP distillation**:
- **Teacher:** Frozen `bert-base-uncased` (768-dim `[CLS]` representation).
- **Student:** Lightweight 2-layer `transformer` (256-dim, 4 heads) or `bilstm` trained from scratch on SST-2 sentiment classification.
- **Why sweep `τ` and `β` here:** Text `[CLS]` embeddings from a pretrained transformer have different isotropy/cosine-similarity geometry than CNN gap features. Comparing `(τ, β)` sensitivity on SST-2 against CIFAR-100 shows whether optimal contrastive temperature and weighting generalize across modalities.
- **Note:** Leave `RUN_NLP_SWEEP = False` if you only want the CIFAR-100 sweep; set `RUN_NLP_SWEEP = True` to run, plot, and interpret the NLP sweep (including a `beta=0` no-CRD baseline if desired via `INCLUDE_BASELINE = True`).

In [ ]:
# Step 7 (Optional): Run NLP CRD sweep on SST-2 with tqdm + visualization & interpretation
import itertools
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

RUN_NLP_SWEEP = False      # Set True to run NLP experiments
STUDENT_TYPE = "transformer"  # "transformer" or "bilstm"
NLP_EPOCHS = 10

if RUN_NLP_SWEEP:
    nlp_taus = [0.03, 0.07, 0.14, 0.30]
    nlp_betas = [0.2, 0.4, 0.8, 1.2]
    nlp_save_root = "/content/drive/MyDrive/crd_sweep_results/nlp_crd"
    os.makedirs(nlp_save_root, exist_ok=True)

    # Optional CE-only baseline (beta=0) to measure pure CRD gain
    configs = [(0.07, 0.0)] + list(itertools.product(nlp_taus, nlp_betas))
    summary_rows = []

    for trial, (tau, beta) in enumerate(tqdm(configs, desc="NLP CRD sweep"), start=1):
        run_dir = os.path.join(nlp_save_root, f"{STUDENT_TYPE}_tau{tau}_beta{beta}")
        log_csv = os.path.join(run_dir, f"sst2_{STUDENT_TYPE}_trial{trial}_log.csv")

        if not os.path.exists(log_csv):
            cmd = [
                "python", "train_nlp_crd.py",
                "--student_type", STUDENT_TYPE,
                "--task", "sst2",
                "--nce_t", str(tau),
                "--beta", str(beta),
                "--epochs", str(NLP_EPOCHS),
                "--num_workers", "2",
                "--trial", str(trial),
                "--save_dir", run_dir,
            ]
            tqdm.write(f"[NLP {trial}/{len(configs)}] tau={tau}, beta={beta}")
            subprocess.run(cmd, check=True)

        run_df = pd.read_csv(log_csv)
        summary_rows.append({
            "trial": trial,
            "tau": tau,
            "beta": beta,
            "best_val_acc": float(run_df["val_acc"].max()),
            "final_val_acc": float(run_df["val_acc"].iloc[-1]),
            "final_loss_ce": float(run_df["loss_ce"].iloc[-1]),
            "final_loss_crd": float(run_df["loss_crd"].iloc[-1]),
            "log_csv": log_csv,
        })

    nlp_summary = pd.DataFrame(summary_rows)
    nlp_summary.to_csv(os.path.join(nlp_save_root, "nlp_summary.csv"), index=False)

    # Visualize NLP sweep heatmap + CE/CRD loss & val_acc curves
    crd_only = nlp_summary[nlp_summary["beta"] > 0]
    nlp_piv = crd_only.pivot_table(index="tau", columns="beta", values="best_val_acc")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    im = axes[0].imshow(nlp_piv.values, aspect="auto", cmap="viridis")
    fig.colorbar(im, ax=axes[0], label="Best Val Acc (%)")
    axes[0].set_xticks(range(len(nlp_piv.columns)))
    axes[0].set_xticklabels([f"{b:g}" for b in nlp_piv.columns])
    axes[0].set_yticks(range(len(nlp_piv.index)))
    axes[0].set_yticklabels([f"{t:g}" for t in nlp_piv.index])
    axes[0].set_xlabel("Beta (β)")
    axes[0].set_ylabel("Tau (τ)")
    axes[0].set_title(f"SST-2 {STUDENT_TYPE} Best Val Acc (%)")
    for i in range(len(nlp_piv.index)):
        for j in range(len(nlp_piv.columns)):
            v = nlp_piv.values[i, j]
            if not np.isnan(v):
                axes[0].text(j, i, f"{v:.2f}", ha="center", va="center", color="white")

    base_row = nlp_summary[nlp_summary["beta"] == 0].iloc[0]
    best_nlp = crd_only.loc[crd_only["best_val_acc"].idxmax()]
    for label, r in [("CE Baseline (β=0)", base_row), (f"Best CRD (τ={best_nlp['tau']:g}, β={best_nlp['beta']:g})", best_nlp)]:
        cdf = pd.read_csv(r["log_csv"])
        axes[1].plot(cdf["epoch"], cdf["val_acc"], marker="o", label=f"{label}: {r['best_val_acc']:.2f}%")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation Acc (%)")
    axes[1].set_title("SST-2: Best CRD vs No-CRD Baseline")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    nlp_fig_path = os.path.join(nlp_save_root, "nlp_sweep_analysis.png")
    plt.savefig(nlp_fig_path, dpi=150)
    plt.show()

    gain = best_nlp["best_val_acc"] - base_row["best_val_acc"]
    print(f"CE baseline best val_acc : {base_row['best_val_acc']:.2f}%")
    print(f"Best NLP CRD config     : tau={best_nlp['tau']:g}, beta={best_nlp['beta']:g} -> {best_nlp['best_val_acc']:.2f}% (gain: {gain:+.2f}%)")
    if gain > 0.5:
        print("Interpretation: Contrastive [CLS] distillation meaningfully improves SST-2 generalization over CE alone.")
    elif gain >= 0:
        print("Interpretation: Modest/neutral CRD gain on SST-2; 2-layer student may already saturate from supervised labels or need a larger student/harder GLUE task.")
    else:
        print("Interpretation: CRD hurt val_acc vs CE baseline — likely beta is too high or projection alignment interferes with task classification.")
    display(nlp_summary.sort_values("best_val_acc", ascending=False))

## Step 8: Package & Download Final Results
**Where your outputs live:**
- **Google Drive (permanent):** `/MyDrive/crd_sweep_results/` (`results.csv`, `heatmap_tau_beta.png`, `sensitivity_analysis.png`, `learning_curves_comparison.png`, `log_*.txt`, `crd_sweep_results.zip`, and optional `nlp_crd/`).
- **Direct browser download:** Running the cell below downloads `crd_sweep_results.zip` directly to your computer.

In [ ]:
# Step 8: Zip all results + plots (and NLP results if run) and trigger download
import os
import shutil
from google.colab import files

bundle_dir = "/content/final_crd_bundle"
os.makedirs(bundle_dir, exist_ok=True)

drive_results = "/content/drive/MyDrive/crd_sweep_results"
vision_out = os.path.join(bundle_dir, "vision_sweep_results")
os.makedirs(vision_out, exist_ok=True)
for src_dir in [drive_results, "sweep_results"]:
    if os.path.exists(src_dir):
        for fname in os.listdir(src_dir):
            if fname.endswith((".csv", ".png", ".txt")):
                shutil.copy2(os.path.join(src_dir, fname), os.path.join(vision_out, fname))

nlp_root = "/content/drive/MyDrive/crd_sweep_results/nlp_crd"
if os.path.exists(nlp_root):
    shutil.copytree(nlp_root, os.path.join(bundle_dir, "nlp_crd"), dirs_exist_ok=True)

archive_path = shutil.make_archive("/content/crd_sweep_results", "zip", bundle_dir)
shutil.copy2(archive_path, "/content/drive/MyDrive/crd_sweep_results/crd_sweep_results.zip")
print("Saved archive:", archive_path)

files.download(archive_path)